# CASAS Aruba Smart Home Dataset - Dementia Detection
## Stage 1: Understanding & Loading CASAS Data

**Important:** This project uses the **CASAS Aruba** dataset from https://casas.wsu.edu/datasets/
- Single elderly resident
- Months of longitudinal sensor data
- Labeled activities (Sleep, Meal_Preparation, Eating, etc.)
- Best suited for dementia pattern detection

**Data Format:**
```
2010-11-04 00:03:50.209589  M009  ON   Sleep_Out_of_Bed
2010-11-04 00:03:53.121010  M009  OFF
2010-11-04 00:04:01.209589  M010  ON
```
Each row: `timestamp | sensor_id | state | activity_label (optional)`

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import timedelta
import os

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("✓ Libraries imported successfully!")

## Stage 1: Load CASAS Aruba Dataset

The CASAS data format has variable columns:
- Timestamp, sensor ID, state are always present
- Activity label is only on some rows (begin/end markers)
- Some rows have "begin"/"end" meta-tokens we need to skip

In [ ]:
def load_casas(filepath):
    """
    Load raw CASAS .txt file into a clean DataFrame.
    Handles the variable number of columns (activity label is optional).
    """
    rows = []
    with open(filepath, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 4:
                continue
            timestamp = parts[0] + ' ' + parts[1]
            sensor_id = parts[2]
            state     = parts[3]
            # Activity label is only present on some rows (begin/end markers)
            activity  = parts[4] if len(parts) > 4 else None
            # CASAS uses "begin"/"end" as a 5th token — skip those meta-tokens
            if activity in ('begin', 'end'):
                activity = parts[5] if len(parts) > 5 else None
            rows.append({
                'timestamp': pd.to_datetime(timestamp),
                'sensor':    sensor_id,
                'state':     state,
                'activity':  activity
            })

    df = pd.DataFrame(rows)
    df = df.sort_values('timestamp').reset_index(drop=True)
    return df

print("✓ load_casas() function defined")

In [ ]:
# Load the CASAS Aruba dataset
# Adjust the path to where you saved the downloaded file
filepath = 'aruba/data'  # or './data/aruba' or wherever you extracted it

# Check if file exists
if os.path.exists(filepath):
    print(f"Loading data from: {filepath}")
    df = load_casas(filepath)
    print(f"✓ Data loaded successfully!")
    print(f"  Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
    print(f"  Date range: {df['timestamp'].min()} to {df['timestamp'].max()}")
    print(f"  Duration: {(df['timestamp'].max() - df['timestamp'].min()).days} days")
else:
    print(f"❌ File not found: {filepath}")
    print("\nPlease download the CASAS Aruba dataset from:")
    print("https://casas.wsu.edu/datasets/")
    print("\nExtract it and update the filepath variable above.")

## Explore the Data Structure

In [ ]:
# Display first 10 rows
print("First 10 rows of data:")
df.head(10)

In [ ]:
# Check unique activities (these are the labeled behaviors)
activities = df['activity'].dropna().unique()
print(f"Found {len(activities)} unique activities:")
for i, activity in enumerate(activities, 1):
    count = (df['activity'] == activity).sum()
    print(f"  {i:2d}. {activity:<30} ({count:,} occurrences)")

In [ ]:
# Check unique sensors
sensors = df['sensor'].unique()
print(f"Found {len(sensors)} unique sensors:")
print(f"Sensors: {sorted(sensors)}")

In [ ]:
# Data info
print("Dataset Information:")
print(f"  Total events: {len(df):,}")
print(f"  Labeled events (with activity): {df['activity'].notna().sum():,}")
print(f"  Unlabeled events: {df['activity'].isna().sum():,}")
print(f"\nSensor states: {df['state'].unique()}")
print(f"\nMissing values:")
print(df.isnull().sum())

In [ ]:
# Time-based analysis
df['hour'] = df['timestamp'].dt.hour
df['day_of_week'] = df['timestamp'].dt.dayofweek
df['date'] = df['timestamp'].dt.date

print("Temporal Distribution:")
print(f"  Events per day (average): {len(df) / df['date'].nunique():.0f}")
print(f"  Most active hour: {df['hour'].mode()[0]}:00")
print(f"  Least active hour: {df.groupby('hour').size().idxmin()}:00")

## Data Visualizations

Now let's visualize sensor activity patterns and temporal distributions

---
## Stage 2: Feature Engineering (Most Important!)

**Goal:** Transform raw sensor events into daily behavioral features that capture routine patterns.

**Key Concept:** Each row = one day in the resident's life. Features capture:
- Activity patterns (kitchen/bedroom/bathroom visits)
- Temporal routines (wake/sleep times, meal windows)
- Dementia-specific indicators (night wandering, meal skipping, sedentary behavior)
- Safety risks (stove usage)

In [ ]:
# Define sensor categories based on CASAS Aruba sensor map
# These map physical sensors to room/function types

KITCHEN_SENSORS  = {'M008', 'M009', 'M010', 'M011', 'M012', 'M013',
                    'M014', 'M015', 'M016', 'M017', 'M018', 'M019',
                    'M020', 'M021', 'M022', 'M023'}
BEDROOM_SENSORS  = {'M001', 'M002', 'M003', 'M004', 'M005'}
BATHROOM_SENSORS = {'M026', 'M027', 'M028', 'M029', 'M030', 'M031'}
DOOR_SENSORS     = {'D001', 'D002', 'D003', 'D004'}

# Stove sensor — critical for gas safety monitoring
STOVE_SENSOR = 'M014'   # adjust to actual stove sensor ID

print("✓ Sensor categories defined")